# BBH eval: turnstyle's deterministic dispatch

[\![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdonaldson/turnstyle/blob/main/experiments/bbh_eval_colab.ipynb)

Routes [BIG-Bench-Hard](https://github.com/suzgunmirac/BIG-Bench-Hard) prompts through
turnstyle's typed `Task` ADT (`dispatch.run`) and measures accuracy. The symbolic tasks
solve **with no model at all** — `parse → compute → answer` — so this whole eval runs in
seconds. Self-contained: installs `turnstyle` from PyPI, fetches BBH from GitHub.

## Setup

In [ ]:
\!pip install -q turnstyle

In [ ]:
import requests
BBH = "https://raw.githubusercontent.com/suzgunmirac/BIG-Bench-Hard/main/bbh"
def load_bbh(task):
    return requests.get(f"{BBH}/{task}.json").json()["examples"]   # [{input, target}, ...]

## Deterministic eval — no model needed

Each task maps to a deterministic `Task` variant. `dispatch.run()` parses the prompt,
solves it symbolically, and returns an `Answer` (or `Abstain` when it can't — which it
does *gracefully*, never a wrong guess).

In [ ]:
from turnstyle.dispatch import run, Ctx, Abstain

TASKS = {
    "multistep_arithmetic_two":        "Arithmetic",
    "boolean_expressions":             "Boolean",
    "dyck_languages":                  "Dyck",
    "word_sorting":                    "Sorting",
    "navigate":                        "Spatial",
    "web_of_lies":                     "TruthChain",
    "logical_deduction_three_objects": "Ordering",
    "logical_deduction_five_objects":  "Ordering",
    "logical_deduction_seven_objects": "Ordering",
    "date_understanding":              "DateCalc",
}

ctx = Ctx()   # no model: deterministic variants only
print(f"{'task':32s} {'variant':11s} {'acc':>7s} {'abstain':>8s} {'n':>5s}")
print("-" * 68)
tot_correct = tot = 0
for task, variant in TASKS.items():
    data = load_bbh(task)
    correct = abstain = 0
    for ex in data:
        r = run(ex["input"], ctx)
        if isinstance(r, Abstain):
            abstain += 1
        elif r.text == ex["target"]:
            correct += 1
    n = len(data); tot_correct += correct; tot += n
    print(f"{task:32s} {variant:11s} {correct/n:6.1%} {abstain/n:7.1%} {n:5d}")
print("-" * 68)
print(f"{'TOTAL solved deterministically':32s} {'':11s} {tot_correct/tot:6.1%}   ({tot_correct}/{tot})")

## Reading the results

- The **symbolic tasks** (arithmetic, boolean, dyck, word-sorting, navigate, web-of-lies,
  logical-deduction) solve at ~**100%** — the parser fully determines the answer, no model.
- **`date_understanding`** is a deterministic *floor* (~44%): `parse_bbh_date` handles the
  clean cases and **abstains** on the long tail of NL date phrasings (holiday names,
  distractor dates, "day before yesterday"). The abstains are the right failure mode — a
  router escalates them rather than guessing. The principled fix is an LLM-extraction
  `DateExtract` variant (deferred).
- Nothing here loaded a model. That's the floor turnstyle gives you for free.

## Optional — multiple choice with a probe (needs the model)

The MC tasks (snarks, etc.) can't be solved by parsing — they route through a per-option
**probe**. `fit_choice` fits one (via `autoprobe`) and attaches it. This cell loads
SmolLM2 and fits the snarks probe — **slow; use a GPU runtime** (Runtime → Change runtime
type → GPU).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from turnstyle import DispatchTurnstyle
from turnstyle.dispatch import Answer

MODEL = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
tok = AutoTokenizer.from_pretrained(MODEL)
mdl = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=dtype).to(device).eval()

snarks = load_bbh("snarks")
dt = DispatchTurnstyle(mdl, tok, device)
print("fitting snarks probe (autoprobe sweep)…")
print(dt.fit_choice(snarks).reason)

correct = 0
for ex in snarks:
    ans = dt.parse(ex["input"])               # routes MC -> ChoiceProbe pick
    if isinstance(ans, Answer) and ans.text == ex["target"]:
        correct += 1
print(f"snarks via ChoiceProbe: {correct}/{len(snarks)} (in-sample; honest number is the CV above)")

## See also

Step-by-step trace of a single solver (parse → enrich → solve → ground):
[walkthrough notebook](https://colab.research.google.com/github/jdonaldson/turnstyle/blob/main/experiments/dispatch_walkthrough_colab.ipynb)
· `experiments/dispatch_walkthrough_colab.ipynb`.